# `ai_parse_document` — Multi-Format Extraction & Accuracy Analysis

## Overview
This notebook benchmarks Databricks `ai_parse_document` (v2.0) against **4 document types**:

| File | Format | Key challenge |
|---|---|---|
| `sample_dashboard.png` | Image / PNG | Figure + text extraction from a raster image |
| `sample_invoice.pdf` | PDF (structured) | Tables, key-value pairs, numeric accuracy |
| `sample_presentation.pptx` | PowerPoint | Multi-slide layout, mixed text/image |
| `old_articles.pdf` | Scanned PDF | Handwritten / low-resolution OCR |

## Accuracy Metrics
For each file we measure:
- **Element coverage** — types of elements detected (text, table, figure, …)
- **Error rate** — `error_status` from the function output
- **Completeness score** — ratio of non-empty elements vs total pages
- **Confidence proxy** — average element confidence when available
- **Format-specific checks** — e.g. table row count for invoice, slide count for PPTX

## 1. Configuration

In [0]:
%run ../_config/config_unity_catalog

In [0]:
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Paths ──────────────────────────────────────────────────────────────────
PATH_VOLUME   = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_IMAGES   = f"/Volumes/{catalog}/{schema}/raw_data/parsed_images"
PATH_RESULTS  = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"

# Create output directories if they don't exist
dbutils.fs.mkdirs(PATH_IMAGES)
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Files to analyse ───────────────────────────────────────────────────────
FILES = {
    "dashboard_png" : "sample_dashboard.png",
    "Invoice_jpg"   : "Invoice.jpg",
    "AccidentStatement_pdf" : "AccidentStatement.pdf",
    #"old_articles"  : "old_articles.pdf", 
}

print(f"Volume path   : {PATH_VOLUME}")
print(f"Images output : {PATH_IMAGES}")
print(f"Results output: {PATH_RESULTS}")
print(f"Files to parse: {list(FILES.values())}")

## 2. Parse All Documents with `ai_parse_document` v2.0

We use:
- `version = '2.0'` — latest schema (Jan 2026), based on `elements` array (replaces the legacy `pages` array)
- `descriptionElementTypes = '*'` — in v2.0, `'*'` and `'figure'` produce **the same behaviour**: AI descriptions are generated for **figures only** (descriptions for tables and text blocks are not yet supported)
- `imageOutputPath` — persist rendered page images to the volume for visual inspection or multi-modal RAG

In [0]:
parse_results = {}
parse_timings = {}

for label, filename in FILES.items():
    file_path = f"{PATH_VOLUME}/{filename}"
    print(f"\nParsing [{label}] → {filename}")
    
    t0 = time.time()
    sql = f"""
        WITH parsed AS (
            SELECT
                path,
                ai_parse_document(
                    content,
                    map(
                        'version',                '2.0',
                        'imageOutputPath',        '{PATH_IMAGES}/{label}/',
                        'descriptionElementTypes','*'
                    )
                ) AS parsed
            FROM READ_FILES('{file_path}', format => 'binaryFile')
        )
        SELECT
            path,
            to_json(parsed)                         AS parsed_json,
            parsed:error_status                     AS error_status,
            parsed:metadata:version                   AS version,
            size(cast(parsed:document:elements as array<variant>)) AS element_count
        FROM parsed
    """
    
    df = spark.sql(sql)
    row = df.collect()[0]
    elapsed = round(time.time() - t0, 2)
    parse_timings[label] = elapsed
    
    parsed_data = json.loads(row["parsed_json"])
    parse_results[label] = {
        "filename"      : filename,
        "path"          : row["path"],
        "error_status"  : row["error_status"],
        "version"       : row["version"],
        "element_count" : row["element_count"],
        "parsed"        : parsed_data,
        "elapsed_s"     : elapsed,
    }
    
    status = "" if row["error_status"] is None else f"  {row['error_status']}"
    print(f"   {status}  |  version : {row['version']}  |  elements: {row['element_count']}  |  {elapsed}s")

print("\nAll documents parsed.")

## 3. Element-Level Analysis

Extract the `elements` array from each document and build a flat DataFrame to analyse element types, confidence, and content.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

# ── Explicit schema: avoids CANNOT_DETERMINE_TYPE when all values are None ─
# Note: ai_parse_document v2.0 uses 'content' (not 'text') and 'page_id' is
# nested inside bbox[0]['page_id'], not a top-level field.
SCHEMA = StructType([
    StructField("file_label",      StringType(),  False),
    StructField("filename",        StringType(),  False),
    StructField("element_type",    StringType(),  False),
    StructField("page_number",     IntegerType(), True),
    StructField("has_content",     BooleanType(), False),
    StructField("has_description", BooleanType(), False),
    StructField("has_confidence",  BooleanType(), False),
    StructField("confidence",      DoubleType(),  True),   # always None in v2.0 — typed explicitly
    StructField("content_length",  IntegerType(), False),
])

# ── Flatten elements from all files ────────────────────────────────────────
all_elements = []

for label, result in parse_results.items():
    doc      = result["parsed"].get("document", {})
    elements = doc.get("elements", [])

    for elem in elements:
        # page_id is nested inside bbox list in v2.0
        bbox        = elem.get("bbox", [])
        page_number = int(bbox[0]["page_id"]) if bbox and "page_id" in bbox[0] else None

        # v2.0 uses 'content', not 'text'
        content     = elem.get("content") or ""
        description = elem.get("description") or ""
        conf_raw    = elem.get("confidence")          # None in current v2.0 model

        all_elements.append((
            label,
            result["filename"],
            str(elem.get("type", "unknown")),
            page_number,
            bool(content.strip()),
            bool(description.strip()),
            conf_raw is not None,
            float(conf_raw) if conf_raw is not None else None,
            len(content),
        ))

df_elements = spark.createDataFrame(all_elements, schema=SCHEMA)
df_elements.createOrReplaceTempView("elements")

print(f"Total elements across all files: {df_elements.count()}")
display(df_elements.groupBy("file_label", "element_type").count().orderBy("file_label", "count", ascending=[True, False]))

## 4. Per-File Accuracy Scorecard

We compute a **composite accuracy score** based on:

| Sub-score | Weight | Description |
|---|---|---|
| `no_error`         | 30% | `error_status` is null |
| `element_coverage` | 30% | At least 1 element extracted per page |
| `text_fill_rate`   | 20% | % of elements containing non-empty text |
| `avg_confidence`   | 20% | Average confidence score (when available) |

In [0]:
scorecards = []

for label, result in parse_results.items():
    elements = result["parsed"].get("document", {}).get("elements", [])
    page_count = result.get("page_count") or 1
    elem_count = len(elements)
    
    # Sub-scores ─────────────────────────────────────────────────────────────
    # 1. No error
    no_error = 1.0 if result["error_status"] is None else 0.0

    # 2. Element coverage: at least 1 element per page (capped at 1.0)
    element_coverage = min(elem_count / max(int(page_count), 1), 1.0)

    # 3. Text fill rate: % of elements with non-empty text
    if elem_count > 0:
        text_elems = sum(1 for e in elements if e.get("text", "").strip())
        text_fill_rate = text_elems / elem_count
    else:
        text_fill_rate = 0.0

    # 4. Average confidence (default 0.75 when not provided by model)
    confidences = [float(e["confidence"]) for e in elements if "confidence" in e]
    avg_confidence = sum(confidences) / len(confidences) if confidences else 0.75

    # Composite score
    score = (
        0.30 * no_error +
        0.30 * element_coverage +
        0.20 * text_fill_rate +
        0.20 * avg_confidence
    ) * 100

    # Element type breakdown
    type_counts = {}
    for e in elements:
        t = e.get("type", "unknown")
        type_counts[t] = type_counts.get(t, 0) + 1

    scorecards.append({
        "file_label"       : label,
        "filename"         : result["filename"],
        "error_status"     : result["error_status"] or "none",
        "page_count"       : int(page_count),
        "element_count"    : elem_count,
        "no_error"         : round(no_error * 100, 1),
        "element_coverage" : round(element_coverage * 100, 1),
        "text_fill_rate"   : round(text_fill_rate * 100, 1),
        "avg_confidence"   : round(avg_confidence * 100, 1),
        "composite_score"  : round(score, 1),
        "elapsed_s"        : result["elapsed_s"],
        "element_types"    : json.dumps(type_counts),
    })

df_scorecard = spark.createDataFrame([Row(**s) for s in scorecards])
df_scorecard.createOrReplaceTempView("scorecard")

display(
    df_scorecard.select(
        "filename", "page_count", "element_count",
        "no_error", "element_coverage", "text_fill_rate", "avg_confidence",
        "composite_score", "elapsed_s"
    ).orderBy(F.col("composite_score").desc())
)

## 5. Format-Specific Deep Dives

### 5a. Invoice PDF — Table & Key-Value Extraction

In [0]:
invoice_elements = parse_results["invoice_pdf"]["parsed"].get("document", {}).get("elements", [])
tables  = [e for e in invoice_elements if e.get("type") == "table"]
# Key-value pairs: text elements whose content contains ':'
kv_text = [e for e in invoice_elements if e.get("type") == "text" and ":" in (e.get("content") or "")]

print(f"Invoice analysis")
print(f"   Tables detected        : {len(tables)}")
print(f"   Key-value text blocks  : {len(kv_text)}")

if tables:
    for i, t in enumerate(tables):
        desc    = (t.get("description") or "(no description)")[:100]
        content = t.get("content") or ""
        # Count <tr> occurrences as a proxy for row count in HTML table content
        row_count = content.count("<tr>") - 1  # subtract header row
        print(f"   Table {i+1}: ~{row_count} data rows | {desc}")

print("\nSample key-value extractions:")
for kv in kv_text[:5]:
    print(f"   {(kv.get('content') or '').strip()[:120]}")

### 5b. PPTX — Slide-by-Slide Coverage

In [0]:
pptx_elements = parse_results["presentation"]["parsed"].get("document", {}).get("elements", [])

slide_data = {}
for e in pptx_elements:
    p = e.get("page", 0)
    if p not in slide_data:
        slide_data[p] = {"text": 0, "figure": 0, "table": 0, "other": 0}
    t = e.get("type", "other")
    if t in slide_data[p]:
        slide_data[p][t] += 1
    else:
        slide_data[p]["other"] += 1

rows_slides = [
    Row(slide=int(p), text=v["text"], figure=v["figure"], table=v["table"], other=v["other"])
    for p, v in sorted(slide_data.items())
]

if rows_slides:
    df_slides = spark.createDataFrame(rows_slides)
    print(f"📽️ PPTX: {len(slide_data)} slides detected")
    display(df_slides)
else:
    print("⚠️ No slide-level data found.")

### 5c. Scanned PDF (old_articles) — OCR Quality Indicators

In [0]:
scan_elements = parse_results["old_articles"]["parsed"].get("document", {}).get("elements", [])

# Heuristics for OCR quality
import re
garbled_pattern = re.compile(r'[^\x00-\x7F]{3,}|[\x00-\x08\x0b\x0e-\x1f]{2,}')

total_text_chars = 0
garbled_blocks   = 0
empty_text       = 0
samples          = []

for e in scan_elements:
    txt = e.get("text", "")
    total_text_chars += len(txt)
    if not txt.strip():
        empty_text += 1
    elif garbled_pattern.search(txt):
        garbled_blocks += 1
    if txt.strip() and len(samples) < 3:
        samples.append(txt.strip()[:200])

n = len(scan_elements) or 1
ocr_health = round((1 - (garbled_blocks + empty_text) / n) * 100, 1)

print("📰 Scanned PDF — OCR Quality Report")
print(f"   Total elements     : {n}")
print(f"   Empty text blocks  : {empty_text}  ({round(empty_text/n*100,1)}%)")
print(f"   Garbled blocks     : {garbled_blocks}  ({round(garbled_blocks/n*100,1)}%)")
print(f"   Total chars read   : {total_text_chars:,}")
print(f"   OCR health score   : {ocr_health}% ({'🟢 Good' if ocr_health > 75 else '🟡 Medium' if ocr_health > 50 else '🔴 Poor'})")
print("\n📝 Sample extracted text:")
for i, s in enumerate(samples, 1):
    print(f"   [{i}] {s}")

### 5d. PNG Dashboard — Figure Description Quality

In [0]:
png_elements = parse_results["dashboard_png"]["parsed"].get("document", {}).get("elements", [])

figures = [e for e in png_elements if e.get("type") in ("figure", "image")]
text_e  = [e for e in png_elements if e.get("type") == "text"]

print("🖼️ PNG Dashboard analysis")
print(f"   Figure elements   : {len(figures)}")
print(f"   Text elements     : {len(text_e)}")

print("\n AI-generated descriptions (figures only — v2.0 limitation):")
for fig in figures[:3]:
    desc = fig.get("description", "(none)").strip()
    print(f"   ▸ {desc[:250]}")

print("\n📝 Text extracted from image:")
for t in text_e[:5]:
    print(f"   ▸ {t.get('text','').strip()[:150]}")

## 6. Global Summary Dashboard

In [0]:
print("=" * 65)
print(" ai_parse_document — ACCURACY SUMMARY")
print("=" * 65)

header = f"{'File':<30} {'Pages':>5} {'Elems':>6} {'Score':>7} {'Time':>7}"
print(header)
print("-" * 65)

for s in sorted(scorecards, key=lambda x: -x["composite_score"]):
    bar_len = int(s["composite_score"] / 5)
    bar = "█" * bar_len + "░" * (20 - bar_len)
    grade = "🟢" if s["composite_score"] >= 80 else "🟡" if s["composite_score"] >= 55 else "🔴"
    print(f"{s['filename']:<30} {s['page_count']:>5} {s['element_count']:>6} {s['composite_score']:>6.1f}% {s['elapsed_s']:>6.1f}s  {grade}")
    print(f"  {bar}  no_err={s['no_error']}% cov={s['element_coverage']}% fill={s['text_fill_rate']}% conf={s['avg_confidence']}%")
    print()

avg = round(sum(s["composite_score"] for s in scorecards) / len(scorecards), 1)
print(f"\n{'OVERALL AVERAGE SCORE':<40} {avg:>6.1f}%")
print("=" * 65)

## 7. Save Results to Delta Table

In [0]:
TABLE_NAME = f"{catalog}.{schema}.ai_parse_accuracy_results"
run_ts = datetime.now().isoformat()

df_final = df_scorecard.withColumn("run_timestamp", F.lit(run_ts))

(
    df_final.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

print(f"✅ Results saved to: {TABLE_NAME}")
print(f"   Run timestamp: {run_ts}")
display(spark.table(TABLE_NAME).orderBy(F.col("run_timestamp").desc(), F.col("composite_score").desc()))

## 8. Interpretation & Recommendations

### Reading the scores

| Score | Grade | Meaning |
|---|---|---|
| ≥ 80% | 🟢 Good | Reliable extraction — ready for production |
| 55–79% | 🟡 Medium | Usable but review edge-cases |
| < 55% | 🔴 Poor | Needs pre-processing or alternative approach |

### Expected results per format

- **Invoice PDF** → typically 🟢 — structured layout, dense text/table, high element coverage
- **PPTX** → typically 🟢/🟡 — good slide extraction, images may lack text
- **PNG Dashboard** → typically medium — figures well described by AI (via `descriptionElementTypes`), but native text extraction from raster images can be limited
- **Scanned PDF** → typically medium/poor — depends heavily on scan resolution and handwriting legibility

> **v2.0 note:** `descriptionElementTypes='*'` and `descriptionElementTypes='figure'` are equivalent — AI descriptions are generated for **figures only**. Table and text descriptions are not yet supported. Set `descriptionElementTypes=''` to skip descriptions entirely and reduce cost.

### Tips for improving scanned document results

```python
# Pre-process low-quality scans before parsing:
# 1. Increase DPI to ≥ 300 using PIL / OpenCV
# 2. Apply binarisation / noise removal
# 3. Correct skew before calling ai_parse_document
from PIL import Image, ImageFilter
img = Image.open(scan_path).convert('L')  # Grayscale
img = img.filter(ImageFilter.SHARPEN)
img.save(preprocessed_path, dpi=(300, 300))
```